# EHRSHOT数据集分析 - 内科24小时内住院患者筛选

## 目标
1. 筛选内科患者（通过care_site_name科室名称）
2. 筛选住院24小时以内的患者
3. 每个步骤报告筛选掉的数量


In [46]:
import pandas as pd
import numpy as np
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 步骤2: 读取care_site.csv（科室名称映射表）

In [47]:
print("=" * 80)
print("步骤2: 读取care_site.csv（科室名称映射表）")
print("=" * 80)

# 读取care_site.csv
care_site_path = '/home/bingkun_zhao/ehrshot_analysis/data/care_site.csv'
care_site_df = pd.read_csv(care_site_path)

print(f"\ncare_site记录数: {len(care_site_df):,}")
print(f"\n列名: {care_site_df.columns.tolist()}")
print(f"\n样本数据:")
print(care_site_df.head(10))

# 查看有科室名称的记录
care_site_with_name = care_site_df[care_site_df['care_site_name'].notna()].copy()
print(f"\n有科室名称的记录数: {len(care_site_with_name):,}")
print(f"\n科室名称分布（前30个）:")
print(care_site_with_name['care_site_name'].value_counts().head(30))

步骤2: 读取care_site.csv（科室名称映射表）

care_site记录数: 3,343

列名: ['care_site_id', 'care_site_name', 'place_of_service_concept_id', 'location_id', 'care_site_source_value', 'place_of_service_source_value', 'trace_id', 'unit_id', 'load_table_id']

样本数据:
   care_site_id care_site_name  place_of_service_concept_id  location_id  \
0        528678            NaN                            0      6616644   
1        527678            NaN                            0      8499918   
2        529520            NaN                            0      9504702   
3        527957            NaN                            0      7460559   
4        529669            NaN                            0      7517402   
5        529509            NaN                            0     10003767   
6        530101            NaN                            0     10541572   
7        529992            NaN                            0     11460798   
8        528939            NaN                            0      9018493 

## 步骤3: 读取visit_occurrence.csv

In [48]:
print("=" * 80)
print("步骤3: 读取visit_occurrence.csv")
print("=" * 80)

# 读取visit_occurrence.csv
visit_occ_path = '/home/bingkun_zhao/ehrshot_analysis/data/visit_occurrence.csv'
visit_occ_df = pd.read_csv(visit_occ_path)

print(f"\nvisit_occurrence.csv记录数: {len(visit_occ_df):,}")
print(f"\n列名: {visit_occ_df.columns.tolist()}")
print(f"\n样本数据（前5行）:")
print(visit_occ_df.head(5))

# 检查care_site_id列
if 'care_site_id' in visit_occ_df.columns:
    print(f"\n✓ 发现care_site_id列")
    print(f"care_site_id非空记录数: {visit_occ_df['care_site_id'].notna().sum():,}")
    print(f"唯一care_site_id数: {visit_occ_df['care_site_id'].nunique()}")
    
    # 查看care_site_id分布
    print(f"\ncare_site_id样本（前20个）:")
    print(visit_occ_df['care_site_id'].value_counts().head(20))
else:
    print("\n⚠ 没有care_site_id列")

# 转换时间列 - 使用正确的列名（大写DATETIME）
visit_occ_df['visit_start_datetime'] = pd.to_datetime(visit_occ_df['visit_start_DATETIME'])
visit_occ_df['visit_end_datetime'] = pd.to_datetime(visit_occ_df['visit_end_DATETIME'])
visit_occ_df['duration_hours'] = (visit_occ_df['visit_end_datetime'] - visit_occ_df['visit_start_datetime']).dt.total_seconds() / 3600

print(f"\n时长统计（小时）:")
print(visit_occ_df['duration_hours'].describe())

步骤3: 读取visit_occurrence.csv

visit_occurrence.csv记录数: 944,132

列名: ['visit_occurrence_id', 'person_id', 'visit_concept_id', 'visit_start_DATE', 'visit_start_DATETIME', 'visit_end_DATE', 'visit_end_DATETIME', 'visit_type_concept_id', 'provider_id', 'care_site_id', 'visit_source_value', 'visit_source_concept_id', 'admitting_source_concept_id', 'admitting_source_value', 'discharge_to_concept_id', 'discharge_to_source_value', 'preceding_visit_occurrence_id', 'trace_id', 'unit_id', 'load_table_id']

样本数据（前5行）:
   visit_occurrence_id  person_id  visit_concept_id visit_start_DATE  \
0             59759168  115970965              9202       2014-03-02   
1             59477020  115970926              9202       2014-02-02   
2             59689438  115972391              9202       2014-03-12   
3             58959195  115970926              9202       2014-01-09   
4             37256524  115971873                 0       2009-10-18   

  visit_start_DATETIME visit_end_DATE   visit_end_DATETI

## 步骤4: 对齐三个数据源

In [49]:
print("=" * 80)
print("步骤3: 读取visit_occurrence.csv")
print("=" * 80)

# 读取visit_occurrence.csv
visit_occ_path = '/home/bingkun_zhao/ehrshot_analysis/data/visit_occurrence.csv'
visit_occ_df = pd.read_csv(visit_occ_path)

print(f"\nvisit_occurrence.csv记录数: {len(visit_occ_df):,}")
print(f"\n列名: {visit_occ_df.columns.tolist()}")
print(f"\n样本数据（前5行）:")
print(visit_occ_df.head(5))

# 检查care_site_id列
if 'care_site_id' in visit_occ_df.columns:
    print(f"\n✓ 发现care_site_id列")
    print(f"care_site_id非空记录数: {visit_occ_df['care_site_id'].notna().sum():,}")
    print(f"唯一care_site_id数: {visit_occ_df['care_site_id'].nunique()}")
    
    # 查看care_site_id分布
    print(f"\ncare_site_id样本（前20个）:")
    print(visit_occ_df['care_site_id'].value_counts().head(20))
else:
    print("\n⚠ 没有care_site_id列")

# 转换时间列 - 使用日期（不考虑时分秒）
visit_occ_df['visit_start_date'] = pd.to_datetime(visit_occ_df['visit_start_DATE'])
visit_occ_df['visit_end_date'] = pd.to_datetime(visit_occ_df['visit_end_DATE'])

# 计算住院天数（按日期差）
visit_occ_df['duration_days'] = (visit_occ_df['visit_end_date'] - visit_occ_df['visit_start_date']).dt.days

print(f"\n住院天数统计（按日期计算）:")
print(visit_occ_df['duration_days'].describe())

print(f"\n住院天数分布:")
print(f"  0天（当天出院）: {(visit_occ_df['duration_days'] == 0).sum():,}")
print(f"  1天: {(visit_occ_df['duration_days'] == 1).sum():,}")
print(f"  2天及以上: {(visit_occ_df['duration_days'] >= 2).sum():,}")
print(f"  负值（异常）: {(visit_occ_df['duration_days'] < 0).sum():,}")

步骤3: 读取visit_occurrence.csv

visit_occurrence.csv记录数: 944,132

列名: ['visit_occurrence_id', 'person_id', 'visit_concept_id', 'visit_start_DATE', 'visit_start_DATETIME', 'visit_end_DATE', 'visit_end_DATETIME', 'visit_type_concept_id', 'provider_id', 'care_site_id', 'visit_source_value', 'visit_source_concept_id', 'admitting_source_concept_id', 'admitting_source_value', 'discharge_to_concept_id', 'discharge_to_source_value', 'preceding_visit_occurrence_id', 'trace_id', 'unit_id', 'load_table_id']

样本数据（前5行）:
   visit_occurrence_id  person_id  visit_concept_id visit_start_DATE  \
0             59759168  115970965              9202       2014-03-02   
1             59477020  115970926              9202       2014-02-02   
2             59689438  115972391              9202       2014-03-12   
3             58959195  115970926              9202       2014-01-09   
4             37256524  115971873                 0       2009-10-18   

  visit_start_DATETIME visit_end_DATE   visit_end_DATETI

## 步骤5: 使用care_site_id筛选内科患者

In [50]:
print("=" * 80)
print("步骤5: 使用care_site_id筛选内科患者")
print("=" * 80)

# 将care_site_id转换为字符串以便匹配
visit_occ_df['care_site_id_str'] = visit_occ_df['care_site_id'].fillna(0).astype(int).astype(str)

# 筛选内科患者（care_site_id在内科科室列表中）
internal_visits = visit_occ_df[visit_occ_df['care_site_id_str'].isin(internal_care_site_ids)].copy()

print(f"\n筛选前: {len(visit_occ_df):,} 条visit_occurrence记录")
print(f"筛选后: {len(internal_visits):,} 条内科visit记录")
print(f"筛选掉: {len(visit_occ_df) - len(internal_visits):,} 条记录")
print(f"保留比例: {len(internal_visits)/len(visit_occ_df)*100:.2f}%")

if len(internal_visits) > 0:
    # 添加科室名称
    internal_visits['care_site_name'] = internal_visits['care_site_id_str'].map(care_site_id_to_name)
    
    print(f"\n内科患者数: {internal_visits['person_id'].nunique():,}")
    print(f"内科visit数: {internal_visits['visit_occurrence_id'].nunique():,}")
    
    print(f"\n内科科室分布:")
    print(internal_visits['care_site_name'].value_counts())
    
    print(f"\n就诊类型分布:")
    if 'visit_concept_id' in internal_visits.columns:
        print(internal_visits['visit_concept_id'].value_counts().head(10))
else:
    print("\n⚠ 未找到内科visit记录")
    print("可能原因:")
    print("  1. care_site_id的值与care_site.csv不匹配")
    print("  2. 内科关键词没有匹配到正确的科室")

步骤5: 使用care_site_id筛选内科患者

筛选前: 944,132 条visit_occurrence记录
筛选后: 152,494 条内科visit记录
筛选掉: 791,638 条记录
保留比例: 16.15%

内科患者数: 4,164
内科visit数: 152,494

内科科室分布:
INTERNAL MEDICINE                             33366
CARDIOLOGY                                    32890
GASTROENTEROLOGY                              20421
NEPHROLOGY                                    18223
HEMATOLOGY                                    15174
ENDOCRINOLOGY                                  9597
NEUROLOGY                                      9568
INFECTIOUS DISEASES                            4365
RHEUMATOLOGY                                   4188
GENERAL INTERNAL MEDICINE                      1820
REPRODUCTIVE ENDOCRINOLOGY AND INFERTILITY     1798
CARDIOLOGY LAB                                  778
CARDIAC REHABILITATION                          306
Name: care_site_name, dtype: int64

就诊类型分布:
38004193    35895
38004515    26624
581477      25548
581458      21966
9202        15825
5083        15329
0            9498

## 步骤6: 筛选住院患者

In [53]:
print("=" * 80)
print("步骤6: 筛选住院患者")
print("=" * 80)

if len(internal_visits) > 0:
    # 筛选住院患者
    # OMOP CDM中，visit_concept_id = 9201 表示住院（Inpatient Visit）
    # 或者使用visit_concept_id = 262 (Emergency Room and Inpatient Visit)
    inpatient_concept_ids = [9201, 262]  # Inpatient Visit, ER and Inpatient Visit
    
    if 'visit_concept_id' in internal_visits.columns:
        internal_inpatient_visits = internal_visits[
            internal_visits['visit_concept_id'].isin(inpatient_concept_ids)
        ].copy()
        
        print(f"\n使用visit_concept_id筛选住院患者")
        print(f"住院concept_id: {inpatient_concept_ids}")
    else:
        # 如果没有visit_concept_id，尝试其他方法
        print("\n⚠ 没有visit_concept_id列，尝试其他方法")
        internal_inpatient_visits = internal_visits.copy()
    
    print(f"\n筛选前: {len(internal_visits):,} 条内科visit记录")
    print(f"筛选后: {len(internal_inpatient_visits):,} 条内科住院记录")
    print(f"筛选掉: {len(internal_visits) - len(internal_inpatient_visits):,} 条记录")
    
    if len(internal_visits) > 0:
        print(f"保留比例: {len(internal_inpatient_visits)/len(internal_visits)*100:.2f}%")
    
    if len(internal_inpatient_visits) > 0:
        print(f"\n内科住院患者数: {internal_inpatient_visits['person_id'].nunique():,}")
        
        print(f"\n科室分布:")
        print(internal_inpatient_visits['care_site_name'].value_counts())
        
    else:
        print("\n⚠ 未找到内科住院记录")
else:
    print("\n⚠ 没有内科visit记录，无法筛选住院患者")
    internal_inpatient_visits = pd.DataFrame()

步骤6: 筛选住院患者

使用visit_concept_id筛选住院患者
住院concept_id: [9201, 262]

筛选前: 152,494 条内科visit记录
筛选后: 1,776 条内科住院记录
筛选掉: 150,718 条记录
保留比例: 1.16%

内科住院患者数: 834

科室分布:
CARDIOLOGY          1456
GASTROENTEROLOGY     319
NEPHROLOGY             1
Name: care_site_name, dtype: int64


## 步骤7: 筛选24小时内住院

In [54]:
print("=" * 80)
print("步骤7: 筛选住院2天以内（duration_days < 2）")
print("=" * 80)

if len(internal_inpatient_visits) > 0:
    # 筛选住院天数<2天的患者（0天或1天）
    inpatient_visits_2days = internal_inpatient_visits[internal_inpatient_visits['duration_days'] < 2].copy()

    print(f"\n筛选前: {len(internal_inpatient_visits):,} 条内科住院记录")
    print(f"筛选后: {len(inpatient_visits_2days):,} 条2天以内住院记录")
    print(f"筛选掉: {len(internal_inpatient_visits) - len(inpatient_visits_2days):,} 条记录")
    print(f"保留比例: {len(inpatient_visits_2days)/len(internal_inpatient_visits)*100:.2f}%")

    print(f"\n最终患者数: {inpatient_visits_2days['person_id'].nunique():,}")
    print(f"最终visit数: {inpatient_visits_2days['visit_occurrence_id'].nunique():,}")
    
    print(f"\n住院天数统计:")
    print(inpatient_visits_2days['duration_days'].describe())
    
    print(f"\n住院天数分布:")
    print(f"  0天（当天出院）: {(inpatient_visits_2days['duration_days'] == 0).sum():,} ({(inpatient_visits_2days['duration_days'] == 0).sum()/len(inpatient_visits_2days)*100:.1f}%)")
    print(f"  1天: {(inpatient_visits_2days['duration_days'] == 1).sum():,} ({(inpatient_visits_2days['duration_days'] == 1).sum()/len(inpatient_visits_2days)*100:.1f}%)")
    if (inpatient_visits_2days['duration_days'] < 0).sum() > 0:
        print(f"  负值（异常）: {(inpatient_visits_2days['duration_days'] < 0).sum():,} ({(inpatient_visits_2days['duration_days'] < 0).sum()/len(inpatient_visits_2days)*100:.1f}%)")
    
    print(f"\n科室分布:")
    print(inpatient_visits_2days['care_site_name'].value_counts())
    
    final_visits = inpatient_visits_2days
else:
    print("\n⚠ 没有内科住院记录，无法筛选2天以内")
    final_visits = pd.DataFrame()

步骤7: 筛选住院2天以内（duration_days < 2）

筛选前: 1,776 条内科住院记录
筛选后: 1,405 条2天以内住院记录
筛选掉: 371 条记录
保留比例: 79.11%

最终患者数: 714
最终visit数: 1,405

住院天数统计:
count    1405.000000
mean        0.032740
std         0.178019
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: duration_days, dtype: float64

住院天数分布:
  0天（当天出院）: 1,359 (96.7%)
  1天: 46 (3.3%)

科室分布:
CARDIOLOGY          1085
GASTROENTEROLOGY     319
NEPHROLOGY             1
Name: care_site_name, dtype: int64


In [55]:
print("=" * 80)
print("筛选流程总结")
print("=" * 80)

summary_data = {
    '步骤': [
        '1. codes.parquet',
        '2. care_site.csv',
        '3. 内科科室',
        '4. visit_occurrence.csv',
        '5. 内科visit',
        '6. 内科住院',
        '7. 内科2天以内住院'
    ],
    '记录数/数量': [
        f"{len(codes_df):,} 条记录",
        f"{len(care_site_df):,} 条记录",
        f"{len(internal_care_sites):,} 个科室",
        f"{len(visit_occ_df):,} 条记录",
        f"{len(internal_visits):,} 条记录" if 'internal_visits' in locals() and len(internal_visits) > 0 else 'N/A',
        f"{len(internal_inpatient_visits):,} 条记录" if 'internal_inpatient_visits' in locals() and len(internal_inpatient_visits) > 0 else 'N/A',
        f"{len(final_visits):,} 条记录" if len(final_visits) > 0 else 'N/A'
    ],
    '患者数': [
        '-',
        '-',
        '-',
        f"{visit_occ_df['person_id'].nunique():,}",
        f"{internal_visits['person_id'].nunique():,}" if 'internal_visits' in locals() and len(internal_visits) > 0 else 'N/A',
        f"{internal_inpatient_visits['person_id'].nunique():,}" if 'internal_inpatient_visits' in locals() and len(internal_inpatient_visits) > 0 else 'N/A',
        f"{final_visits['person_id'].nunique():,}" if len(final_visits) > 0 else 'N/A'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

print("\n" + "=" * 80)
print("筛选条件")
print("=" * 80)
print("✓ 内科科室: 通过care_site.csv的care_site_name匹配内科关键词")
print("  - INTERNAL MEDICINE, CARDIOLOGY, GASTROENTEROLOGY")
print("  - NEPHROLOGY, ENDOCRINOLOGY, HEMATOLOGY")
print("  - NEUROLOGY, INFECTIOUS DISEASES等")
print("✓ 内科患者: visit_occurrence.csv的care_site_id在内科科室列表中")
print("✓ 住院患者: visit_concept_id = 9201 (Inpatient) 或 262 (ER+Inpatient)")
print("✓ 2天以内: duration_days < 2（按日期计算，不考虑时分秒）")
print("  - 0天: 入院日期 = 出院日期（当天出院）")
print("  - 1天: 出院日期 - 入院日期 = 1天")

print("\n" + "=" * 80)
print("数据源对齐")
print("=" * 80)
print("care_site.csv → care_site_id → visit_occurrence.csv → person_id → ehrshot.csv")

筛选流程总结


                     步骤      记录数/数量   患者数
       1. codes.parquet  95,414 条记录     -
       2. care_site.csv   3,343 条记录     -
                3. 内科科室     453 个科室     -
4. visit_occurrence.csv 944,132 条记录 6,710
             5. 内科visit 152,494 条记录 4,164
                6. 内科住院   1,776 条记录   834
            7. 内科2天以内住院   1,405 条记录   714

筛选条件
✓ 内科科室: 通过care_site.csv的care_site_name匹配内科关键词
  - INTERNAL MEDICINE, CARDIOLOGY, GASTROENTEROLOGY
  - NEPHROLOGY, ENDOCRINOLOGY, HEMATOLOGY
  - NEUROLOGY, INFECTIOUS DISEASES等
✓ 内科患者: visit_occurrence.csv的care_site_id在内科科室列表中
✓ 住院患者: visit_concept_id = 9201 (Inpatient) 或 262 (ER+Inpatient)
✓ 2天以内: duration_days < 2（按日期计算，不考虑时分秒）
  - 0天: 入院日期 = 出院日期（当天出院）
  - 1天: 出院日期 - 入院日期 = 1天

数据源对齐
care_site.csv → care_site_id → visit_occurrence.csv → person_id → ehrshot.csv


# 详细数据分析
## 分析1: 科室详细分布

In [56]:
print("=" * 80)
print("保存结果")
print("=" * 80)

import os
output_dir = '/home/bingkun_zhao/ehrshot_analysis/output'
os.makedirs(output_dir, exist_ok=True)

if len(final_visits) > 0:
    # 保存最终筛选结果
    final_visits.to_csv(f'{output_dir}/internal_medicine_2days_visits.csv', index=False)
    print(f"✓ 已保存: {output_dir}/internal_medicine_2days_visits.csv")
    print(f"  记录数: {len(final_visits):,}")

    # 保存患者ID列表
    patient_list = pd.DataFrame({
        'person_id': sorted(final_visits['person_id'].unique())
    })
    patient_list.to_csv(f'{output_dir}/internal_medicine_2days_patient_ids.csv', index=False)
    print(f"✓ 已保存: {output_dir}/internal_medicine_2days_patient_ids.csv")
    print(f"  患者数: {len(patient_list):,}")

    # 保存内科科室列表
    care_sites_summary = internal_care_sites[['care_site_id', 'care_site_name']].copy()
    care_sites_summary.to_csv(f'{output_dir}/internal_medicine_care_sites.csv', index=False)
    print(f"✓ 已保存: {output_dir}/internal_medicine_care_sites.csv")
    print(f"  科室数: {len(care_sites_summary):,}")
    
    # 保存科室分布统计
    if 'care_site_name' in final_visits.columns:
        dept_stats = final_visits.groupby('care_site_name').agg({
            'person_id': 'nunique',
            'visit_occurrence_id': 'nunique',
            'duration_days': ['mean', 'median', 'min', 'max']
        }).reset_index()
        dept_stats.columns = ['care_site_name', 'patient_count', 'visit_count', 
                               'avg_days', 'median_days', 'min_days', 'max_days']
        dept_stats = dept_stats.sort_values('patient_count', ascending=False)
        dept_stats.to_csv(f'{output_dir}/department_statistics.csv', index=False)
        print(f"✓ 已保存: {output_dir}/department_statistics.csv")
        print(f"  科室统计: {len(dept_stats):,} 个科室")
    
    print(f"\n" + "=" * 80)
    print("最终结果")
    print("=" * 80)
    print(f"✓ 内科2天以内住院患者数: {final_visits['person_id'].nunique():,}")
    print(f"✓ 内科2天以内住院记录数: {len(final_visits):,}")
    print(f"✓ 涉及内科科室数: {final_visits['care_site_name'].nunique()}")
    print(f"\n住院天数分布:")
    print(f"  0天（当天出院）: {(final_visits['duration_days'] == 0).sum():,} ({(final_visits['duration_days'] == 0).sum()/len(final_visits)*100:.1f}%)")
    print(f"  1天: {(final_visits['duration_days'] == 1).sum():,} ({(final_visits['duration_days'] == 1).sum()/len(final_visits)*100:.1f}%)")
else:
    print("⚠ 没有数据可保存")

保存结果
✓ 已保存: /home/bingkun_zhao/ehrshot_analysis/output/internal_medicine_2days_visits.csv
  记录数: 1,405
✓ 已保存: /home/bingkun_zhao/ehrshot_analysis/output/internal_medicine_2days_patient_ids.csv
  患者数: 714
✓ 已保存: /home/bingkun_zhao/ehrshot_analysis/output/internal_medicine_care_sites.csv
  科室数: 453
✓ 已保存: /home/bingkun_zhao/ehrshot_analysis/output/department_statistics.csv
  科室统计: 3 个科室

最终结果
✓ 内科2天以内住院患者数: 714
✓ 内科2天以内住院记录数: 1,405
✓ 涉及内科科室数: 3

住院天数分布:
  0天（当天出院）: 1,359 (96.7%)
  1天: 46 (3.3%)


In [57]:
print("=" * 80)
print("数据质量检查：住院天数异常分析")
print("=" * 80)

if len(final_visits) > 0:
    print("\n1. 天数计算逻辑检查")
    print("=" * 80)
    print("计算公式: duration_days = (visit_end_date - visit_start_date).days")
    print("说明: 只考虑日期，不考虑时分秒")
    print("  - 0天: 入院日期 = 出院日期（当天出院）")
    print("  - 1天: 出院日期 - 入院日期 = 1天")
    
    # 查看原始时间数据
    print("\n2. 原始日期数据样本（前10条）")
    print("=" * 80)
    sample_cols = ['visit_occurrence_id', 'person_id', 'care_site_name', 
                   'visit_start_date', 'visit_end_date', 'duration_days']
    print(final_visits[sample_cols].head(10))
    
    # 统计天数分布
    print("\n3. 住院天数分布统计")
    print("=" * 80)
    print(final_visits['duration_days'].describe())
    
    # 负值天数详细分析
    print("\n4. 负值天数详细分析")
    print("=" * 80)
    negative_duration = final_visits[final_visits['duration_days'] < 0]
    print(f"负值记录数: {len(negative_duration)} ({len(negative_duration)/len(final_visits)*100:.1f}%)")
    
    if len(negative_duration) > 0:
        print(f"\n负值天数范围: {negative_duration['duration_days'].min()} ~ {negative_duration['duration_days'].max()} 天")
        print(f"\n负值天数的科室分布:")
        print(negative_duration['care_site_name'].value_counts())
        
        print(f"\n负值天数样本（前20条）:")
        print(negative_duration[sample_cols].head(20))
        
        print(f"\n负值原因分析:")
        print(f"  可能原因: visit_end_date < visit_start_date（日期颠倒或数据错误）")
    
    # 零天数分析
    print("\n5. 零天数分析（当天出院）")
    print("=" * 80)
    zero_duration = final_visits[final_visits['duration_days'] == 0]
    print(f"零天数记录数: {len(zero_duration)} ({len(zero_duration)/len(final_visits)*100:.1f}%)")
    
    if len(zero_duration) > 0:
        print(f"\n零天数的科室分布:")
        print(zero_duration['care_site_name'].value_counts())
        
        print(f"\n零天数样本（前10条）:")
        print(zero_duration[sample_cols].head(10))
    
    # 1天住院分析
    print("\n6. 1天住院分析")
    print("=" * 80)
    one_day_duration = final_visits[final_visits['duration_days'] == 1]
    print(f"1天住院记录数: {len(one_day_duration)} ({len(one_day_duration)/len(final_visits)*100:.1f}%)")
    
    if len(one_day_duration) > 0:
        print(f"\n1天住院的科室分布:")
        print(one_day_duration['care_site_name'].value_counts())
        
        print(f"\n1天住院样本（前10条）:")
        print(one_day_duration[sample_cols].head(10))
    
    # 各科室天数对比
    print("\n7. 各科室住院天数对比")
    print("=" * 80)
    dept_duration = final_visits.groupby('care_site_name')['duration_days'].agg([
        'count', 'mean', 'median', 'std', 'min', 'max',
        ('负值数', lambda x: (x < 0).sum()),
        ('0天数', lambda x: (x == 0).sum()),
        ('1天数', lambda x: (x == 1).sum())
    ]).round(2)
    
    dept_duration.columns = ['记录数', '平均(天)', '中位(天)', '标准差', '最小(天)', '最大(天)', '负值数', '0天数', '1天数']
    print(dept_duration)
    
    # 建议
    print("\n8. 数据质量建议")
    print("=" * 80)
    print("基于以上分析:")
    if len(negative_duration) > 0:
        print(f"  ⚠ 发现{len(negative_duration)}条负值记录（出院日期早于入院日期），建议排除")
    else:
        print(f"  ✓ 没有负值记录")
    print(f"  ✓ {len(zero_duration)}条0天记录（当天出院）")
    print(f"  ✓ {len(one_day_duration)}条1天记录（住院1天）")
else:
    print("⚠ 没有数据可分析")

数据质量检查：住院天数异常分析

1. 天数计算逻辑检查
计算公式: duration_days = (visit_end_date - visit_start_date).days
说明: 只考虑日期，不考虑时分秒
  - 0天: 入院日期 = 出院日期（当天出院）
  - 1天: 出院日期 - 入院日期 = 1天

2. 原始日期数据样本（前10条）
       visit_occurrence_id  person_id care_site_name visit_start_date  \
83186             61081763  115972358     CARDIOLOGY       2014-05-02   
83187            168483170  115967459     CARDIOLOGY       2020-11-12   
83188            211992076  115970090     CARDIOLOGY       2022-09-17   
83189            213296958  115971454     CARDIOLOGY       2022-12-01   
83190            169958484  115971736     CARDIOLOGY       2020-12-25   
83191            153769730  115967815     CARDIOLOGY       2020-04-01   
83192            168017733  115970457     CARDIOLOGY       2020-11-04   
83193            160378779  115972150     CARDIOLOGY       2020-06-17   
83194            156293234  115968005     CARDIOLOGY       2020-02-28   
83195            166914791  115969043     CARDIOLOGY       2020-11-09   

      visit_end_d

In [58]:
print("=" * 80)
print("导出异常数据（负值天数）")
print("=" * 80)

if len(final_visits) > 0:
    # 筛选负值天数记录
    negative_visits = final_visits[final_visits['duration_days'] < 0].copy()
    
    print(f"\n负值天数记录数: {len(negative_visits)}")
    print(f"涉及患者数: {negative_visits['person_id'].nunique()}")
    
    if len(negative_visits) > 0:
        # 添加更多有用的列
        negative_visits_export = negative_visits[[
            'visit_occurrence_id', 
            'person_id', 
            'care_site_id',
            'care_site_name',
            'visit_concept_id',
            'visit_start_DATE',
            'visit_end_DATE',
            'visit_start_date',
            'visit_end_date',
            'duration_days',
            'visit_source_value',
            'admitting_source_value',
            'discharge_to_source_value'
        ]].copy()
        
        # 按天数排序（从最负到最小负）
        negative_visits_export = negative_visits_export.sort_values('duration_days')
        
        # 保存到文件
        import os
        output_dir = '/home/bingkun_zhao/ehrshot_analysis/output'
        os.makedirs(output_dir, exist_ok=True)
        
        output_file = f'{output_dir}/negative_duration_visits.csv'
        negative_visits_export.to_csv(output_file, index=False)
        
        print(f"\n✓ 已保存负值天数数据: {output_file}")
        print(f"  记录数: {len(negative_visits_export):,}")
        
        # 显示详细信息
        print("\n负值天数统计:")
        print(f"  最小值（最负）: {negative_visits['duration_days'].min()} 天")
        print(f"  最大值（接近0）: {negative_visits['duration_days'].max()} 天")
        print(f"  平均值: {negative_visits['duration_days'].mean():.2f} 天")
        
        # 科室分布
        print("\n负值天数的科室分布:")
        dept_dist = negative_visits['care_site_name'].value_counts()
        for dept, count in dept_dist.items():
            pct = count / len(negative_visits) * 100
            print(f"  {dept}: {count} ({pct:.1f}%)")
        
        # 显示所有记录
        print(f"\n负值天数详细记录（共{len(negative_visits_export)}条）:")
        print("\n" + "=" * 120)
        
        for idx, row in negative_visits_export.iterrows():
            print(f"\n记录 {idx}:")
            print(f"  visit_id: {row['visit_occurrence_id']}")
            print(f"  patient_id: {row['person_id']}")
            print(f"  科室: {row['care_site_name']}")
            print(f"  入院日期: {row['visit_start_date'].date()}")
            print(f"  出院日期: {row['visit_end_date'].date()}")
            print(f"  天数: {row['duration_days']} 天")
            print(f"  问题: 出院日期早于入院日期 {abs(row['duration_days'])} 天")
        
        print("\n" + "=" * 80)
        print("可能的原因分析:")
        print("  1. 数据录入错误：入院和出院日期字段可能被颠倒录入")
        print("  2. 系统数据问题：不同系统的数据同步错误")
        
        print("\n建议:")
        print(f"  - 这{len(negative_visits)}条记录应该被排除在分析之外")
        
    else:
        print("\n✓ 没有负值天数记录，数据质量良好")
else:
    print("⚠ 没有数据可分析")

导出异常数据（负值天数）

负值天数记录数: 0
涉及患者数: 0

✓ 没有负值天数记录，数据质量良好


## 分析2: 患者特征分析

In [59]:
print("=" * 80)
print("分析2: 患者特征分析")
print("=" * 80)

if len(final_visits) > 0:
    # 患者就诊频次分析
    patient_visit_counts = final_visits.groupby('person_id').size()
    
    print("\n患者就诊频次分布:")
    print(f"  总患者数: {len(patient_visit_counts):,}")
    print(f"  总就诊次数: {len(final_visits):,}")
    print(f"  平均每人就诊次数: {patient_visit_counts.mean():.2f}")
    print(f"  中位就诊次数: {patient_visit_counts.median():.0f}")
    print(f"  最多就诊次数: {patient_visit_counts.max()}")
    
    print("\n就诊次数分组:")
    visit_freq_groups = pd.cut(patient_visit_counts, 
                                bins=[0, 1, 2, 3, 5, 10, float('inf')],
                                labels=['1次', '2次', '3次', '4-5次', '6-10次', '10次以上'])
    print(visit_freq_groups.value_counts().sort_index())
    
    # 多次就诊患者分析
    print("\n" + "=" * 80)
    print("多次就诊患者分析（就诊≥3次）")
    print("=" * 80)
    
    frequent_patients = patient_visit_counts[patient_visit_counts >= 3].index
    frequent_visits = final_visits[final_visits['person_id'].isin(frequent_patients)]
    
    print(f"\n多次就诊患者数: {len(frequent_patients):,} ({len(frequent_patients)/len(patient_visit_counts)*100:.1f}%)")
    print(f"多次就诊记录数: {len(frequent_visits):,} ({len(frequent_visits)/len(final_visits)*100:.1f}%)")
    
    if len(frequent_visits) > 0:
        print("\n多次就诊患者的科室分布:")
        print(frequent_visits['care_site_name'].value_counts())
        
        # 查看就诊最多的前10位患者
        top_patients = patient_visit_counts.nlargest(10)
        print("\n就诊次数最多的10位患者:")
        for i, (patient_id, count) in enumerate(top_patients.items(), 1):
            patient_depts = final_visits[final_visits['person_id'] == patient_id]['care_site_name'].value_counts()
            print(f"  {i}. 患者ID {patient_id}: {count}次就诊")
            print(f"     科室分布: {dict(patient_depts)}")
    
    # 跨科室就诊分析
    print("\n" + "=" * 80)
    print("跨科室就诊分析")
    print("=" * 80)
    
    patient_dept_counts = final_visits.groupby('person_id')['care_site_name'].nunique()
    
    print("\n患者涉及科室数分布:")
    print(f"  仅1个科室: {(patient_dept_counts == 1).sum():,} ({(patient_dept_counts == 1).sum()/len(patient_dept_counts)*100:.1f}%)")
    print(f"  2个科室: {(patient_dept_counts == 2).sum():,} ({(patient_dept_counts == 2).sum()/len(patient_dept_counts)*100:.1f}%)")
    print(f"  3个科室: {(patient_dept_counts == 3).sum():,} ({(patient_dept_counts == 3).sum()/len(patient_dept_counts)*100:.1f}%)")
    
    # 跨科室患者的科室组合
    multi_dept_patients = patient_dept_counts[patient_dept_counts > 1].index
    if len(multi_dept_patients) > 0:
        print(f"\n跨科室就诊患者数: {len(multi_dept_patients):,}")
        print("\n常见科室组合（前10）:")
        
        dept_combos = []
        for patient_id in multi_dept_patients:
            depts = sorted(final_visits[final_visits['person_id'] == patient_id]['care_site_name'].unique())
            dept_combos.append(' + '.join(depts))
        
        combo_counts = pd.Series(dept_combos).value_counts().head(10)
        for combo, count in combo_counts.items():
            print(f"  {combo}: {count}位患者")
else:
    print("⚠ 没有数据可分析")

分析2: 患者特征分析

患者就诊频次分布:
  总患者数: 714
  总就诊次数: 1,405
  平均每人就诊次数: 1.97
  中位就诊次数: 1
  最多就诊次数: 22

就诊次数分组:
1次       403
2次       169
3次        62
4-5次      50
6-10次     24
10次以上      6
dtype: int64

多次就诊患者分析（就诊≥3次）

多次就诊患者数: 142 (19.9%)
多次就诊记录数: 664 (47.3%)

多次就诊患者的科室分布:
CARDIOLOGY          548
GASTROENTEROLOGY    116
Name: care_site_name, dtype: int64

就诊次数最多的10位患者:
  1. 患者ID 115969648: 22次就诊
     科室分布: {'CARDIOLOGY': 22}
  2. 患者ID 115968515: 21次就诊
     科室分布: {'CARDIOLOGY': 21}
  3. 患者ID 115972076: 18次就诊
     科室分布: {'CARDIOLOGY': 18}
  4. 患者ID 115970918: 15次就诊
     科室分布: {'CARDIOLOGY': 14, 'GASTROENTEROLOGY': 1}
  5. 患者ID 115971397: 14次就诊
     科室分布: {'CARDIOLOGY': 14}
  6. 患者ID 115971757: 11次就诊
     科室分布: {'CARDIOLOGY': 8, 'GASTROENTEROLOGY': 3}
  7. 患者ID 115967843: 9次就诊
     科室分布: {'CARDIOLOGY': 9}
  8. 患者ID 115969949: 9次就诊
     科室分布: {'CARDIOLOGY': 7, 'GASTROENTEROLOGY': 2}
  9. 患者ID 115968403: 8次就诊
     科室分布: {'CARDIOLOGY': 8}
  10. 患者ID 115968871: 8次就诊
     科室分布: {'CARDIOLOGY': 8}

跨科室就

## 分析3: 数据质量总结

In [62]:
print("=" * 80)
print("分析3: 数据质量总结")
print("=" * 80)

if len(final_visits) > 0:
    print("\n" + "=" * 80)
    print("最终筛选结果汇总")
    print("=" * 80)
    
    print(f"\n✓ 总患者数: {final_visits['person_id'].nunique():,}")
    print(f"✓ 总就诊记录数: {len(final_visits):,}")
    print(f"✓ 涉及科室数: {final_visits['care_site_name'].nunique()}")
    print(f"✓ 时间跨度: {final_visits['visit_start_date'].min().date()} ~ {final_visits['visit_start_date'].max().date()}")
    
    print("\n" + "=" * 80)
    print("科室分布详情")
    print("=" * 80)
    
    dept_summary = final_visits.groupby('care_site_name').agg({
        'person_id': 'nunique',
        'visit_occurrence_id': 'count'
    }).rename(columns={'person_id': '患者数', 'visit_occurrence_id': '就诊次数'})
    dept_summary = dept_summary.sort_values('患者数', ascending=False)
    
    print("\n各科室统计:")
    for dept, row in dept_summary.iterrows():
        patient_pct = row['患者数'] / final_visits['person_id'].nunique() * 100
        visit_pct = row['就诊次数'] / len(final_visits) * 100
        print(f"\n{dept}:")
        print(f"  患者数: {row['患者数']:,} ({patient_pct:.1f}%)")
        print(f"  就诊次数: {row['就诊次数']:,} ({visit_pct:.1f}%)")
        print(f"  人均就诊: {row['就诊次数']/row['患者数']:.2f}次")
    
    print("\n" + "=" * 80)
    print("数据质量指标")
    print("=" * 80)
    
    # 计算各项指标
    total_records = len(final_visits)
    negative_count = (final_visits['duration_days'] < 0).sum()
    zero_count = (final_visits['duration_days'] == 0).sum()
    one_day_count = (final_visits['duration_days'] == 1).sum()
    
    print(f"\n住院天数分布:")
    print(f"  负值（数据异常）: {negative_count} ({negative_count/total_records*100:.1f}%)")
    print(f"  0天（当天出院）: {zero_count} ({zero_count/total_records*100:.1f}%)")
    print(f"  1天: {one_day_count} ({one_day_count/total_records*100:.1f}%)")
    
    # 缺失值检查
    print(f"\n关键字段完整性:")
    key_fields = ['person_id', 'visit_occurrence_id', 'care_site_id', 'care_site_name', 
                  'visit_start_date', 'visit_end_date', 'duration_days']
    
    for field in key_fields:
        if field in final_visits.columns:
            missing = final_visits[field].isna().sum()
            complete_rate = (1 - missing/len(final_visits)) * 100
            print(f"  {field}: {complete_rate:.1f}% 完整")
    
    print("\n" + "=" * 80)
    print("关键发现")
    print("=" * 80)
    
    print("\n1. 科室分布:")
    if 'CARDIOLOGY' in dept_summary.index:
        print(f"   - CARDIOLOGY（心内科）: {dept_summary.loc['CARDIOLOGY', '患者数']}位患者 ({dept_summary.loc['CARDIOLOGY', '患者数']/final_visits['person_id'].nunique()*100:.1f}%)")
    if 'GASTROENTEROLOGY' in dept_summary.index:
        print(f"   - GASTROENTEROLOGY（消化内科）: {dept_summary.loc['GASTROENTEROLOGY', '患者数']}位患者 ({dept_summary.loc['GASTROENTEROLOGY', '患者数']/final_visits['person_id'].nunique()*100:.1f}%)")
    if 'NEPHROLOGY' in dept_summary.index:
        print(f"   - NEPHROLOGY（肾内科）: {dept_summary.loc['NEPHROLOGY', '患者数']}位患者")
    
    print("\n2. 住院天数特征:")
    print(f"   - 0天（当天出院）: {zero_count}条记录 ({zero_count/total_records*100:.1f}%)")
    print(f"   - 1天: {one_day_count}条记录 ({one_day_count/total_records*100:.1f}%)")
    if negative_count > 0:
        print(f"   - 存在{negative_count}条负值记录，可能是数据录入问题")
    
    print("\n3. 患者就诊模式:")
    patient_visit_counts = final_visits.groupby('person_id').size()
    single_visit = (patient_visit_counts == 1).sum()
    multi_visit = (patient_visit_counts > 1).sum()
    print(f"   - 单次就诊患者: {single_visit} ({single_visit/len(patient_visit_counts)*100:.1f}%)")
    print(f"   - 多次就诊患者: {multi_visit} ({multi_visit/len(patient_visit_counts)*100:.1f}%)")
    print(f"   - 最多就诊次数: {patient_visit_counts.max()}次")
    
    print("\n" + "=" * 80)
    print("数据保存位置")
    print("=" * 80)
    print(f"\n输出目录: /home/bingkun_zhao/ehrshot_analysis/output/")
    print(f"\n已保存文件:")
    print(f"  1. internal_medicine_2days_visits.csv - 完整就诊记录")
    print(f"  2. internal_medicine_2days_patient_ids.csv - 患者ID列表")
    print(f"  3. internal_medicine_care_sites.csv - 内科科室列表")
    print(f"  4. department_statistics.csv - 科室统计")
else:
    print("⚠ 没有数据可分析")

分析3: 数据质量总结

最终筛选结果汇总

✓ 总患者数: 714
✓ 总就诊记录数: 1,405
✓ 涉及科室数: 3
✓ 时间跨度: 2007-12-22 ~ 2023-02-08

科室分布详情

各科室统计:

CARDIOLOGY:
  患者数: 538 (75.4%)
  就诊次数: 1,085 (77.2%)
  人均就诊: 2.02次

GASTROENTEROLOGY:
  患者数: 205 (28.7%)
  就诊次数: 319 (22.7%)
  人均就诊: 1.56次

NEPHROLOGY:
  患者数: 1 (0.1%)
  就诊次数: 1 (0.1%)
  人均就诊: 1.00次

数据质量指标

住院天数分布:
  负值（数据异常）: 0 (0.0%)
  0天（当天出院）: 1359 (96.7%)
  1天: 46 (3.3%)

关键字段完整性:
  person_id: 100.0% 完整
  visit_occurrence_id: 100.0% 完整
  care_site_id: 100.0% 完整
  care_site_name: 100.0% 完整
  visit_start_date: 100.0% 完整
  visit_end_date: 100.0% 完整
  duration_days: 100.0% 完整

关键发现

1. 科室分布:
   - CARDIOLOGY（心内科）: 538位患者 (75.4%)
   - GASTROENTEROLOGY（消化内科）: 205位患者 (28.7%)
   - NEPHROLOGY（肾内科）: 1位患者

2. 住院天数特征:
   - 0天（当天出院）: 1359条记录 (96.7%)
   - 1天: 46条记录 (3.3%)

3. 患者就诊模式:
   - 单次就诊患者: 403 (56.4%)
   - 多次就诊患者: 311 (43.6%)
   - 最多就诊次数: 22次

数据保存位置

输出目录: /home/bingkun_zhao/ehrshot_analysis/output/

已保存文件:
  1. internal_medicine_2days_visits.csv - 完整就诊记录
  2. internal_medicine